# Hands-on Activity: Quantizing a Transformer Model
### SC 17, Module 1 — Apply Activity (25 min)

**Scenario:** A small startup has an accurate Transformer-based text classifier running in a research notebook, but it takes too long per request to meet a real-time response requirement. Before investing in quantization-aware training, the team wants a quick, numbers-backed answer: how much latency and size improvement does the cheapest form of compression — post-training dynamic quantization — actually deliver, and at what accuracy cost.

Complete the TODOs in each task cell below. Do not delete the cell structure — the grader expects Task 1, Task 2, and Task 3 in separate, labeled cells.


## Environment Setup

Run this once, in a terminal, before opening this notebook — it creates an isolated environment with exactly the packages this activity needs:

```bash
conda create -n sc17-quant python=3.11 -y
conda activate sc17-quant
pip install -q ipykernel transformers "optimum-onnx[onnxruntime]" datasets onnxruntime numpy
python -m ipykernel install --user --name sc17-quant --display-name "SC17 Quantization"
```

Then in Jupyter: click the kernel name in the top-right corner and select **"SC17 Quantization"** before running any cells below.


## Task 1: Export and Quantize the Model (8 min)

**Context:** This task builds the compression pipeline itself: exporting the PyTorch model to ONNX, then wrapping it with Optimum's `ORTQuantizer` to produce a quantized INT8 variant.

**Practice:**
1. Complete the ONNX export step and confirm the exported directory contains a valid `model.onnx` file. What happens if you try to quantize a model that was never exported to ONNX first?
2. Instantiate `ORTQuantizer` from the exported directory, then build a dynamic `AutoQuantizationConfig`. Why does dynamic quantization not require a calibration dataset the way static quantization does?
3. Run the quantization step and confirm a new INT8 ONNX model file exists in the output directory. Compare the file sizes of the original ONNX export and the quantized version.


In [ ]:
import os
import platform
from optimum.onnxruntime import ORTModelForSequenceClassification, ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig

# This activity uses the same CPU-architecture detection covered in the
# HOW reading: quantization presets differ between ARM (Apple Silicon)
# and x86 (Intel/AMD) hardware.
if platform.machine() in ("arm64", "aarch64"):
    onnx_arch = "arm64"
else:
    onnx_arch = "avx2"

MODEL_CHECKPOINT_PATH = "distilbert-base-uncased-finetuned-sst-2-english"  # provided checkpoint
ONNX_OUTPUT_DIR = None        # TODO: set an output directory, e.g. "onnx_fp32"
QUANTIZED_OUTPUT_DIR = None   # TODO: set a separate output directory for the INT8 model

# TODO: Load and export the provided checkpoint to ONNX
onnx_model = ORTModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT_PATH,
    export=True,
)
onnx_model.save_pretrained(ONNX_OUTPUT_DIR)

# TODO: Instantiate an ORTQuantizer from the exported ONNX model directory


# TODO: Build a dynamic quantization config with AutoQuantizationConfig
#       (is_static=False). Use `getattr(AutoQuantizationConfig, onnx_arch)`
#       so the config matches whichever architecture was detected above.


# TODO: Call .quantize() on the quantizer, passing the config and
#       QUANTIZED_OUTPUT_DIR as the save_dir


**Success Checklist:**
- [ ] A valid FP32 ONNX model exists in the export directory
- [ ] A separate, valid INT8 ONNX model exists in the quantized output directory
- [ ] The quantized file is measurably smaller than the FP32 export


## Task 2: Build the Benchmarking Loop (10 min)

**Context:** A single latency number is not trustworthy on its own — this task builds a benchmarking loop with a proper warm-up phase and captures the p50 and p95 latency percentiles for both the baseline and quantized models.

**Practice:**
1. Implement the warm-up phase first, running at least 10 untimed iterations before any measurement begins. Why does skipping warm-up produce a misleadingly high first-run latency measurement?
2. Implement the timed loop, recording each individual forward-pass latency rather than only a single total elapsed time. Why does recording per-call latency matter for computing p95 specifically?
3. Run `benchmark_model()` against both the FP32 baseline model and the INT8 quantized model from Task 1, using the same tokenized sample inputs for both. Record all four resulting numbers: FP32 p50, FP32 p95, INT8 p50, INT8 p95.


In [ ]:
import time
import numpy as np
from transformers import AutoTokenizer

SAMPLE_TEXTS = [
    "The product arrived on time and works exactly as described.",
    "Customer support was unhelpful and the issue was never resolved.",
    "Shipping was fast but the packaging was damaged.",
]

def benchmark_model(session, tokenized_inputs, warmup_iters=10, timed_iters=50):
    # TODO: Run `warmup_iters` forward passes without timing them, to let
    #       the runtime reach a steady state (JIT warm-up, cache population)


    latencies = []
    # TODO: Run `timed_iters` forward passes, recording the wall-clock time
    #       of each individual forward pass in `latencies`


    # TODO: Compute and return the p50 and p95 latency from `latencies`
    #       using numpy's percentile function
    pass

# TODO: Create ONNX Runtime InferenceSession objects for both the FP32 model
#       (ONNX_OUTPUT_DIR) and the INT8 model (QUANTIZED_OUTPUT_DIR)


# TODO: Build a tokenized input dict for one sample text (e.g. SAMPLE_TEXTS[0]),
#       filtered to only the input names each session actually expects
#       (compare AutoTokenizer's output keys against session.get_inputs())


# TODO: Call benchmark_model() once for the FP32 session and once for the
#       INT8 session, using the same input text for both, then print
#       FP32 p50/p95, INT8 p50/p95, and the percentage improvement at each
#       percentile


**Success Checklist:**
- [ ] The warm-up phase runs a minimum of 10 iterations before any timed measurement begins
- [ ] The timed loop records individual per-call latencies, not a single aggregate total
- [ ] p50 and p95 are computed correctly from the recorded latency array
- [ ] Both the baseline and quantized models were benchmarked using identical input data


## Task 3: Write the Trade-off Report (7 min)

**Context:** Numbers alone do not make the case for or against shipping a compressed model — this task turns the Task 1 and Task 2 results into a short, evidence-based interpretation.

**Practice:**
1. Compare the FP32 and INT8 model file sizes recorded in Task 1. State the approximate percentage reduction in size.
2. Compare the p50 and p95 latency figures recorded in Task 2. State the approximate percentage improvement at each percentile, and note whether the improvement is larger or smaller at p95 than at p50.
3. Run the provided sample text through both models and compare their predicted labels. Based on file size, latency, and any observed prediction differences, write a 150–250 word recommendation on whether this dynamic PTQ result is sufficient for the startup's real-time requirement, or whether static PTQ or QAT should be investigated next.

**Success Checklist:**
- [ ] Size reduction is stated as a specific percentage, calculated from the recorded file sizes
- [ ] Latency improvement is stated separately for p50 and p95, calculated from the recorded benchmark numbers
- [ ] The recommendation is grounded in the specific numbers recorded above, not a generic statement about quantization

---
**Write your 150–250 word trade-off report here:**

*(Replace this text with your analysis — one paragraph covering the size and latency numbers, followed by one paragraph giving your recommendation.)*


In [ ]:
# TODO: Loop over SAMPLE_TEXTS. For each one, run it through both the
#       FP32 session and the INT8 session, take the argmax of each
#       session's output logits as the predicted label, and print
#       whether the two models agree on that sample.
